# ChestX-ray14 평가 (Kaggle): TTA·Ensemble → Calibration → Operating Point → Subgroup → Grad-CAM → External

학습(`kaggle_train.ipynb`)이 끝난 뒤 실행합니다. 모델 추론(GPU)은 이 노트북에서 한 번만 하고,
결과 CSV(`reports/evaluation/`)를 내려받으면 Calibration·Operating Point·Subgroup은 로컬 CPU에서 다시 돌릴 수 있습니다.

**처음 한 번 설정 (오른쪽 패널)**
1. Session options → Accelerator: **GPU T4 x2**(1개만 사용), Internet: **On** (해부학 분할 가중치 다운로드)
2. Add Input → Datasets: **NIH Chest X ray 14 (224x224 resized)** (khanfashee)
3. Add Input → Your Datasets: **chest-xray14-code** (최신 코드로 New Version 올린 것)
4. Add Input → Your Work → **학습 노트북의 마지막 버전 Output** (`outputs/<run_id>/fold_k/best.pt` 20개)
5. (External Validation) Add Input → Datasets: **CheXpert-v1.0-small** (예: `ashery/chexpert`)

**실행**: `RUN_SMOKE_TEST = True`로 셀을 하나씩 실행해 확인 → `False`로 바꾸고 Save & Run All (Commit)
→ 끝나면 Output의 `evaluation_reports.zip`을 내려받아 레포의 `reports/evaluation/`에 풀기

In [ ]:
# ===== 설정 =====
RUN_SMOKE_TEST = True        # True: 이미지 일부로 동작 확인 (몇 분), False: 전체 평가
SMOKE_LIMIT = 200
RUN_GRADCAM = True
RUN_EXTERNAL = True          # CheXpert 입력이 없으면 자동으로 건너뜀
CHEXPERT_SPLIT = "train"     # "train"(NLP 라벨, 환자 단위 표본 아래 수만큼, 보고서 기준) 또는 "valid"(234장, 전문의 합의 라벨)
CHEXPERT_TRAIN_SAMPLE = 5000

In [ ]:
# 1) 코드 가져오기 — 코드 데이터셋(chest-xray14-code)을 /tmp/repo로 복사
#    (Kaggle Input은 읽기 전용이라 쓰기 가능한 /tmp로 복사)
import glob, os, shutil

def find_input(rel, max_depth=5):
    """/kaggle/input 아래에서 rel을 얕게 찾기. recursive glob(**)은 CheXpert·NIH 이미지 수십만 장을
    전부 훑어서 몇 시간씩 멈춘 것처럼 보임 → 폴더 깊이 max_depth까지만 확인"""
    hits = []
    for depth in range(max_depth + 1):
        hits += glob.glob("/kaggle/input/" + "*/" * depth + rel)
    return sorted(set(hits))

found = find_input("src/training/run_queue.py")
print(found)
assert found, "코드 데이터셋을 찾지 못했습니다 (Add Input → Your Datasets → chest-xray14-code 추가 확인)"
assert len(found) == 1, "코드 데이터셋이 여러 개 연결되어 있습니다 (하나만 남기기)"

code_root = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
shutil.rmtree("/tmp/repo", ignore_errors=True)
shutil.copytree(code_root, "/tmp/repo")

%cd /tmp/repo
!ls
!pip install -q pyyaml
!nvidia-smi -L              # Tesla T4 두 줄이 나와야 함

In [ ]:
# 2) 이미지를 Kaggle 서버 디스크(/tmp)로 복사 (내 컴퓨터로 받는 것 아님) — /kaggle/input은 파일마다 읽기 지연이 커서
#    학습 중 GPU가 이미지를 기다림. 처음에 병렬로 한 번 복사해 두면 학습 중 읽기가 빨라짐
#    (메타데이터의 Image Path = images/images-224/<파일명> 구조에 맞춰 복사)
import time
from concurrent.futures import ThreadPoolExecutor

hits = find_input("images-224/00000001_000.png")
assert hits, "00000001_000.png를 찾지 못했습니다 (NIH 데이터셋 추가 여부 확인)"
src_dir = os.path.dirname(hits[0])

RAW = "/tmp/raw"
dst_dir = f"{RAW}/images/images-224"
if os.path.islink(f"{RAW}/images"):                   # 예전 방식(링크)이 남아 있으면 제거
    os.remove(f"{RAW}/images")
os.makedirs(dst_dir, exist_ok=True)

names = [n for n in os.listdir(src_dir) if n.endswith(".png")]
todo = [n for n in names if not os.path.exists(f"{dst_dir}/{n}")]
print(f"이미지 {len(names):,}장 중 {len(todo):,}장 복사 시작")

def copy(name):
    tmp = f"{dst_dir}/{name}.part"                     # 중간에 끊겨도 깨진 파일이 남지 않게
    shutil.copyfile(f"{src_dir}/{name}", tmp)
    os.replace(tmp, f"{dst_dir}/{name}")

t0 = time.time()
with ThreadPoolExecutor(64) as ex:
    for i, _ in enumerate(ex.map(copy, todo), 1):
        if i % 10000 == 0:
            print(f"  {i:,}장 | {i / (time.time() - t0):,.0f}장/s")
print(f"복사 완료: {time.time() - t0:.0f}s")

n_local = len([n for n in os.listdir(dst_dir) if n.endswith(".png")])
print("로컬 이미지 수:", n_local)                     # 112120이어야 함
assert n_local == len(names), "복사된 이미지 수가 다릅니다"

In [ ]:
# 3) 학습 결과(best.pt) 찾기 — 학습 노트북 Output을 Input으로 추가해야 함
ckpts = find_input("outputs/*/fold_*/best.pt")
assert ckpts, "best.pt를 찾지 못했습니다 (Add Input → Your Work → 학습 노트북 Output 추가)"
OUT = sorted({c.split("/outputs/")[0] + "/outputs" for c in ckpts})[0]
print("학습 결과:", OUT, "| best.pt", len(ckpts), "개")
!ls {OUT}

EVAL = "/kaggle/working/reports/evaluation"
LIMIT = f"--limit {SMOKE_LIMIT}" if RUN_SMOKE_TEST else ""
BOOT = "--bootstrap 20" if RUN_SMOKE_TEST else ""
!pip install -q torchxrayvision

## 4) Test + OOF 예측 (GPU, TTA·Ensemble) — 전체 약 15~25분

In [ ]:
!python -m src.evaluation.tta_ensemble --output-dir {OUT} --raw-dir {RAW} --report-dir {EVAL} --num-workers 4 --batch-size 128 {LIMIT}

## 5) Calibration → Operating Point → Subgroup (CPU, 예측 CSV만 사용)

In [ ]:
!python -m src.evaluation.calibration --eval-dir {EVAL}
!python -m src.evaluation.operating_point --eval-dir {EVAL}
!python -m src.evaluation.subgroup --eval-dir {EVAL} {BOOT}

## 6) Grad-CAM: FP/FN 사례, 폐 영역 이탈 분석

In [ ]:
if RUN_GRADCAM:
    !python -m src.xai.gradcam_analysis --eval-dir {EVAL} --output-dir {OUT} --raw-dir {RAW}

## 7) External Validation (CheXpert)

In [ ]:
csvs = [p for p in find_input(f"{CHEXPERT_SPLIT}.csv") if "chex" in p.lower()]
if RUN_EXTERNAL and csvs:
    root = os.path.dirname(csvs[0])
    sample = f"--sample {CHEXPERT_TRAIN_SAMPLE}" if CHEXPERT_SPLIT == "train" else ""
    if RUN_SMOKE_TEST:
        sample = "--sample 100"
    print("CheXpert:", csvs[0])
    !python -m src.evaluation.external_validation --chexpert-csv {csvs[0]} --chexpert-root {root} --output-dir {OUT} --eval-dir {EVAL} --num-workers 4 {sample}
else:
    print("CheXpert 입력이 없어 건너뜀 (Add Input → CheXpert-v1.0-small)")

## 8) 결과 묶기 → Output에서 `evaluation_reports.zip` 다운로드

In [ ]:
import shutil
from pathlib import Path
from IPython.display import Markdown, display
shutil.make_archive("/kaggle/working/evaluation_reports", "zip", EVAL)
for name in ["calibration.md", "tta_comparison.md", "ensemble_comparison.md"]:
    p = Path(EVAL) / name
    if p.exists():
        display(Markdown(p.read_text()))
!ls -la {EVAL}